In [3]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 1) DATOS HISTÓRICOS DE EJEMPLO
#    En un caso real esto vendría de:
#    - una tabla de HC histórico por empleado o por puesto
#    - una tabla financiera por área y año
# ============================================================

hc_hist = pd.DataFrame([
    # ---------------- TI ----------------
    [2020, 'TI', 'Programador Jr', 18],
    [2020, 'TI', 'Programador Sr', 10],
    [2020, 'TI', 'Lider', 3],
    [2020, 'TI', 'Gerente', 1],

    [2021, 'TI', 'Programador Jr', 21],
    [2021, 'TI', 'Programador Sr', 11],
    [2021, 'TI', 'Lider', 3],
    [2021, 'TI', 'Gerente', 1],

    [2022, 'TI', 'Programador Jr', 25],
    [2022, 'TI', 'Programador Sr', 13],
    [2022, 'TI', 'Lider', 4],
    [2022, 'TI', 'Gerente', 1],

    [2023, 'TI', 'Programador Jr', 29],
    [2023, 'TI', 'Programador Sr', 15],
    [2023, 'TI', 'Lider', 4],
    [2023, 'TI', 'Gerente', 2],

    [2024, 'TI', 'Programador Jr', 33],
    [2024, 'TI', 'Programador Sr', 18],
    [2024, 'TI', 'Lider', 5],
    [2024, 'TI', 'Gerente', 2],

    # ---------------- FINANZAS ----------------
    [2020, 'Finanzas', 'Analista', 14],
    [2020, 'Finanzas', 'Especialista', 7],
    [2020, 'Finanzas', 'Coordinador', 2],
    [2020, 'Finanzas', 'Gerente', 1],

    [2021, 'Finanzas', 'Analista', 15],
    [2021, 'Finanzas', 'Especialista', 7],
    [2021, 'Finanzas', 'Coordinador', 2],
    [2021, 'Finanzas', 'Gerente', 1],

    [2022, 'Finanzas', 'Analista', 16],
    [2022, 'Finanzas', 'Especialista', 8],
    [2022, 'Finanzas', 'Coordinador', 2],
    [2022, 'Finanzas', 'Gerente', 1],

    [2023, 'Finanzas', 'Analista', 17],
    [2023, 'Finanzas', 'Especialista', 8],
    [2023, 'Finanzas', 'Coordinador', 3],
    [2023, 'Finanzas', 'Gerente', 1],

    [2024, 'Finanzas', 'Analista', 18],
    [2024, 'Finanzas', 'Especialista', 9],
    [2024, 'Finanzas', 'Coordinador', 3],
    [2024, 'Finanzas', 'Gerente', 1],

    # ---------------- RH ----------------
    [2020, 'RH', 'Analista', 10],
    [2020, 'RH', 'Especialista', 4],
    [2020, 'RH', 'Coordinador', 2],
    [2020, 'RH', 'Gerente', 1],

    [2021, 'RH', 'Analista', 10],
    [2021, 'RH', 'Especialista', 5],
    [2021, 'RH', 'Coordinador', 2],
    [2021, 'RH', 'Gerente', 1],

    [2022, 'RH', 'Analista', 11],
    [2022, 'RH', 'Especialista', 5],
    [2022, 'RH', 'Coordinador', 2],
    [2022, 'RH', 'Gerente', 1],

    [2023, 'RH', 'Analista', 12],
    [2023, 'RH', 'Especialista', 6],
    [2023, 'RH', 'Coordinador', 2],
    [2023, 'RH', 'Gerente', 1],

    [2024, 'RH', 'Analista', 13],
    [2024, 'RH', 'Especialista', 6],
    [2024, 'RH', 'Coordinador', 2],
    [2024, 'RH', 'Gerente', 1],

    # ---------------- LOGISTICA ----------------
    [2020, 'Logistica', 'Analista', 22],
    [2020, 'Logistica', 'Especialista', 12],
    [2020, 'Logistica', 'Coordinador', 4],
    [2020, 'Logistica', 'Gerente', 1],

    [2021, 'Logistica', 'Analista', 24],
    [2021, 'Logistica', 'Especialista', 13],
    [2021, 'Logistica', 'Coordinador', 4],
    [2021, 'Logistica', 'Gerente', 1],

    [2022, 'Logistica', 'Analista', 27],
    [2022, 'Logistica', 'Especialista', 14],
    [2022, 'Logistica', 'Coordinador', 5],
    [2022, 'Logistica', 'Gerente', 1],

    [2023, 'Logistica', 'Analista', 30],
    [2023, 'Logistica', 'Especialista', 16],
    [2023, 'Logistica', 'Coordinador', 5],
    [2023, 'Logistica', 'Gerente', 1],

    [2024, 'Logistica', 'Analista', 33],
    [2024, 'Logistica', 'Especialista', 18],
    [2024, 'Logistica', 'Coordinador', 6],
    [2024, 'Logistica', 'Gerente', 2],

    # ---------------- COMERCIAL ----------------
    [2020, 'Comercial', 'Analista', 16],
    [2020, 'Comercial', 'Especialista', 8],
    [2020, 'Comercial', 'Coordinador', 3],
    [2020, 'Comercial', 'Gerente', 1],

    [2021, 'Comercial', 'Analista', 18],
    [2021, 'Comercial', 'Especialista', 9],
    [2021, 'Comercial', 'Coordinador', 3],
    [2021, 'Comercial', 'Gerente', 1],

    [2022, 'Comercial', 'Analista', 20],
    [2022, 'Comercial', 'Especialista', 10],
    [2022, 'Comercial', 'Coordinador', 3],
    [2022, 'Comercial', 'Gerente', 1],

    [2023, 'Comercial', 'Analista', 22],
    [2023, 'Comercial', 'Especialista', 11],
    [2023, 'Comercial', 'Coordinador', 4],
    [2023, 'Comercial', 'Gerente', 1],

    [2024, 'Comercial', 'Analista', 25],
    [2024, 'Comercial', 'Especialista', 12],
    [2024, 'Comercial', 'Coordinador', 4],
    [2024, 'Comercial', 'Gerente', 1],

    # ---------------- MARKETING ----------------
    [2020, 'Marketing', 'Analista', 9],
    [2020, 'Marketing', 'Especialista', 5],
    [2020, 'Marketing', 'Coordinador', 2],
    [2020, 'Marketing', 'Gerente', 1],

    [2021, 'Marketing', 'Analista', 10],
    [2021, 'Marketing', 'Especialista', 5],
    [2021, 'Marketing', 'Coordinador', 2],
    [2021, 'Marketing', 'Gerente', 1],

    [2022, 'Marketing', 'Analista', 11],
    [2022, 'Marketing', 'Especialista', 6],
    [2022, 'Marketing', 'Coordinador', 2],
    [2022, 'Marketing', 'Gerente', 1],

    [2023, 'Marketing', 'Analista', 12],
    [2023, 'Marketing', 'Especialista', 6],
    [2023, 'Marketing', 'Coordinador', 3],
    [2023, 'Marketing', 'Gerente', 1],

    [2024, 'Marketing', 'Analista', 14],
    [2024, 'Marketing', 'Especialista', 7],
    [2024, 'Marketing', 'Coordinador', 3],
    [2024, 'Marketing', 'Gerente', 1],
], columns=['anio', 'area', 'puesto', 'headcount'])


fin_hist = pd.DataFrame([
    [2020, 'TI',         280,   0,   300, 0.04, 0.03, 0.08, 0.12, 45000, 0.02],
    [2021, 'TI',         320,   0,   340, 0.07, 0.05, 0.10, 0.11, 47000, 0.03],
    [2022, 'TI',         370,   0,   390, 0.08, 0.04, 0.14, 0.12, 50000, 0.04],
    [2023, 'TI',         430,   0,   460, 0.045,0.06, 0.18, 0.13, 54000, 0.05],
    [2024, 'TI',         500,   0,   540, 0.042,0.05, 0.22, 0.14, 58000, 0.05],

    [2020, 'Finanzas',   210,   0,   225, 0.04, 0.03, 0.04, 0.08, 38000, 0.01],
    [2021, 'Finanzas',   225,   0,   240, 0.07, 0.05, 0.05, 0.08, 39500, 0.01],
    [2022, 'Finanzas',   240,   0,   255, 0.08, 0.04, 0.06, 0.09, 41500, 0.02],
    [2023, 'Finanzas',   255,   0,   275, 0.045,0.06, 0.07, 0.09, 44000, 0.02],
    [2024, 'Finanzas',   275,   0,   295, 0.042,0.05, 0.08, 0.08, 46500, 0.02],

    [2020, 'RH',         120,   0,   130, 0.04, 0.03, 0.03, 0.09, 30000, 0.01],
    [2021, 'RH',         130,   0,   138, 0.07, 0.05, 0.04, 0.10, 31500, 0.01],
    [2022, 'RH',         138,   0,   147, 0.08, 0.04, 0.05, 0.09, 33000, 0.01],
    [2023, 'RH',         148,   0,   158, 0.045,0.06, 0.06, 0.09, 35000, 0.02],
    [2024, 'RH',         158,   0,   170, 0.042,0.05, 0.07, 0.08, 37000, 0.02],

    [2020, 'Logistica',  520, 820,  560, 0.04, 0.03, 0.06, 0.16, 26000, 0.03],
    [2021, 'Logistica',  570, 900,  610, 0.07, 0.05, 0.08, 0.15, 27500, 0.03],
    [2022, 'Logistica',  630, 980,  675, 0.08, 0.04, 0.10, 0.14, 29000, 0.04],
    [2023, 'Logistica',  700,1080,  750, 0.045,0.06, 0.12, 0.13, 31000, 0.04],
    [2024, 'Logistica',  780,1180,  835, 0.042,0.05, 0.14, 0.12, 33000, 0.05],

    [2020, 'Comercial',  310,1250,  340, 0.04, 0.03, 0.03, 0.11, 34000, 0.02],
    [2021, 'Comercial',  340,1380,  370, 0.07, 0.05, 0.04, 0.10, 35500, 0.02],
    [2022, 'Comercial',  375,1500,  410, 0.08, 0.04, 0.05, 0.10, 37500, 0.03],
    [2023, 'Comercial',  420,1660,  455, 0.045,0.06, 0.06, 0.09, 40000, 0.03],
    [2024, 'Comercial',  470,1840,  510, 0.042,0.05, 0.08, 0.09, 42500, 0.04],

    [2020, 'Marketing',  150, 420,  170, 0.04, 0.03, 0.03, 0.10, 32000, 0.02],
    [2021, 'Marketing',  165, 460,  185, 0.07, 0.05, 0.04, 0.10, 33500, 0.02],
    [2022, 'Marketing',  180, 510,  205, 0.08, 0.04, 0.05, 0.09, 35500, 0.03],
    [2023, 'Marketing',  198, 570,  225, 0.045,0.06, 0.06, 0.09, 37500, 0.03],
    [2024, 'Marketing',  220, 640,  250, 0.042,0.05, 0.07, 0.08, 39500, 0.04],
], columns=[
    'anio', 'area', 'gasto_area_millones', 'ingresos_area_millones',
    'presupuesto_area_millones', 'inflacion_pct', 'crecimiento_empresa_pct',
    'automatizacion_pct', 'rotacion_pct', 'salario_promedio_mensual',
    'crecimiento_pais_pct'
])


# ============================================================
# 2) UNIR HC Y FINANZAS
# ============================================================
df = hc_hist.merge(fin_hist, on=['anio', 'area'], how='left')

# En caso real, aquí validarías faltantes de merge
if df.isna().sum().sum() > 0:
    raise ValueError("Hay nulos después del merge entre HC y finanzas. Revisa llaves.")


# ============================================================
# 3) FEATURES HISTÓRICAS
#    - HC total por área y año
#    - participación del puesto dentro del área
#    - lags del headcount
#    - target = HC del siguiente año por área y puesto
# ============================================================

# HC total por área/año
hc_total_area = (
    df.groupby(['anio', 'area'], as_index=False)['headcount']
      .sum()
      .rename(columns={'headcount': 'hc_total_area'})
)

df = df.merge(hc_total_area, on=['anio', 'area'], how='left')

# Participación del puesto dentro del HC total del área
df['share_puesto_en_area'] = np.where(
    df['hc_total_area'] > 0,
    df['headcount'] / df['hc_total_area'],
    0
)

# Ordenar para generar lags
df = df.sort_values(['area', 'puesto', 'anio']).reset_index(drop=True)

# Lags por área y puesto
df['hc_lag_1'] = df.groupby(['area', 'puesto'])['headcount'].shift(1)
df['hc_lag_2'] = df.groupby(['area', 'puesto'])['headcount'].shift(2)

# Lag de share por área y puesto
df['share_lag_1'] = df.groupby(['area', 'puesto'])['share_puesto_en_area'].shift(1)

# Target: HC del siguiente año para esa área y puesto
df['hc_target_next_year'] = df.groupby(['area', 'puesto'])['headcount'].shift(-1)

# Como en la vida real, perdemos filas iniciales por los rezagos y finales por el target
df_model = df.dropna(subset=['hc_lag_1', 'hc_lag_2', 'share_lag_1', 'hc_target_next_year']).copy()

# Aseguramos target entero/no negativo
df_model['hc_target_next_year'] = df_model['hc_target_next_year'].astype(int)


# ============================================================
# 4) FEATURES Y TARGET
# ============================================================
target = 'hc_target_next_year'

features = [
    'anio',
    'area',
    'puesto',
    'headcount',
    'hc_lag_1',
    'hc_lag_2',
    'hc_total_area',
    'share_puesto_en_area',
    'share_lag_1',
    'gasto_area_millones',
    'ingresos_area_millones',
    'presupuesto_area_millones',
    'inflacion_pct',
    'crecimiento_empresa_pct',
    'automatizacion_pct',
    'rotacion_pct',
    'salario_promedio_mensual',
    'crecimiento_pais_pct'
]

X = df_model[features].copy()
y = df_model[target].copy()


# ============================================================
# 5) SPLIT TEMPORAL
#    Con los lags y el target, solo quedan años 2022 y 2023.
#    Entrenamos con 2022 y probamos con 2023.
# ============================================================
train_mask = X['anio'] <= 2022
test_mask = X['anio'] == 2023

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

print("Años disponibles en X:", sorted(X['anio'].unique()))
print("Train años:", sorted(X_train['anio'].unique()))
print("Test años:", sorted(X_test['anio'].unique()))
print("Filas train:", len(X_train))
print("Filas test:", len(X_test))

if len(X_test) == 0:
    raise ValueError("No hay registros de prueba. Revisa el corte temporal.")


# ============================================================
# 6) PREPROCESAMIENTO
# ============================================================
categorical_features = ['area', 'puesto']

numeric_features = [
    'anio',
    'headcount',
    'hc_lag_1',
    'hc_lag_2',
    'hc_total_area',
    'share_puesto_en_area',
    'share_lag_1',
    'gasto_area_millones',
    'ingresos_area_millones',
    'presupuesto_area_millones',
    'inflacion_pct',
    'crecimiento_empresa_pct',
    'automatizacion_pct',
    'rotacion_pct',
    'salario_promedio_mensual',
    'crecimiento_pais_pct'
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


# ============================================================
# 7) MODELO
#    PoissonRegressor es más razonable para conteos de HC
# ============================================================
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', PoissonRegressor(alpha=0.10, max_iter=1000))
])

model.fit(X_train, y_train)


# ============================================================
# 8) EVALUACIÓN
# ============================================================
y_pred = model.predict(X_test)
y_pred_round = np.maximum(np.round(y_pred), 0).astype(int)

mae = mean_absolute_error(y_test, y_pred_round)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_round))
r2 = r2_score(y_test, y_pred_round)

print("=== METRICAS TEST ===")
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2  : {r2:,.4f}")

resultado_test = X_test[['anio', 'area', 'puesto', 'headcount']].copy()
resultado_test['real_next_year'] = y_test.values
resultado_test['pred_next_year'] = y_pred_round
resultado_test['error_abs'] = (resultado_test['real_next_year'] - resultado_test['pred_next_year']).abs()

print("\n=== RESULTADOS TEST ===")
print(resultado_test.sort_values(['area', 'puesto']).to_string(index=False))

# Error promedio por área
metricas_area = (
    resultado_test.groupby('area', as_index=False)
    .agg(
        mae_area=('error_abs', 'mean'),
        hc_real_total=('real_next_year', 'sum'),
        hc_pred_total=('pred_next_year', 'sum')
    )
)

print("\n=== ERROR PROMEDIO POR AREA ===")
print(metricas_area.to_string(index=False))


# ============================================================
# 9) PREPARAR ESCENARIO FUTURO
#    Queremos estimar 2025 usando:
#    - el último histórico real (2024)
#    - supuestos financieros 2025 por área
# ============================================================

supuestos_2025 = pd.DataFrame([
    ['TI',         560,   0,   600, 0.040, 0.055, 0.26, 0.13, 62000, 0.025],
    ['Finanzas',   295,   0,   320, 0.040, 0.045, 0.09, 0.08, 48500, 0.025],
    ['RH',         170,   0,   182, 0.040, 0.040, 0.08, 0.08, 38500, 0.025],
    ['Logistica',  860,1260,  920, 0.040, 0.060, 0.16, 0.11, 34500, 0.025],
    ['Comercial',  520,1980,  565, 0.040, 0.055, 0.09, 0.09, 44000, 0.025],
    ['Marketing',  245, 710,  280, 0.040, 0.050, 0.08, 0.08, 41000, 0.025],
], columns=[
    'area', 'gasto_area_millones', 'ingresos_area_millones',
    'presupuesto_area_millones', 'inflacion_pct', 'crecimiento_empresa_pct',
    'automatizacion_pct', 'rotacion_pct', 'salario_promedio_mensual',
    'crecimiento_pais_pct'
])

ultimo_anio = df['anio'].max()

# Tomamos los datos 2024 para cada área/puesto
base_2024 = df[df['anio'] == ultimo_anio].copy()

# Necesitamos también hc_lag_1 y hc_lag_2 reales para armar features 2025
hist_sorted = df.sort_values(['area', 'puesto', 'anio']).copy()

# Armamos lags sobre todo el histórico
hist_sorted['hc_lag_1_full'] = hist_sorted.groupby(['area', 'puesto'])['headcount'].shift(1)
hist_sorted['hc_lag_2_full'] = hist_sorted.groupby(['area', 'puesto'])['headcount'].shift(2)

# Nos quedamos con 2024 ya enriquecido
base_2024 = hist_sorted[hist_sorted['anio'] == ultimo_anio].copy()

# HC total área 2024
hc_total_2024 = (
    base_2024.groupby('area', as_index=False)['headcount']
    .sum()
    .rename(columns={'headcount': 'hc_total_area'})
)

base_2024 = base_2024.drop(columns=['hc_total_area', 'share_puesto_en_area'], errors='ignore')
base_2024 = base_2024.merge(hc_total_2024, on='area', how='left')

base_2024['share_puesto_en_area'] = np.where(
    base_2024['hc_total_area'] > 0,
    base_2024['headcount'] / base_2024['hc_total_area'],
    0
)

# Share lag 1 = share 2023
share_2023 = df[df['anio'] == (ultimo_anio - 1)][['area', 'puesto', 'share_puesto_en_area']].copy()
share_2023 = share_2023.rename(columns={'share_puesto_en_area': 'share_lag_1'})

# Eliminar columna previa si ya existe para evitar sufijos _x / _y
base_2024 = base_2024.drop(columns=['share_lag_1'], errors='ignore')

base_2024 = base_2024.merge(share_2023, on=['area', 'puesto'], how='left')

# Unimos supuestos 2025 por área
escenario_2025 = base_2024[['area', 'puesto', 'headcount', 'hc_lag_1_full', 'hc_lag_2_full', 'hc_total_area',
                            'share_puesto_en_area', 'share_lag_1']].copy()

escenario_2025 = escenario_2025.rename(columns={
    'hc_lag_1_full': 'hc_lag_1',
    'hc_lag_2_full': 'hc_lag_2'
})

escenario_2025 = escenario_2025.merge(supuestos_2025, on='area', how='left')
escenario_2025['anio'] = 2025

# Reordenar columnas exactamente como espera el modelo
escenario_2025 = escenario_2025[features].copy()

# Validación básica
if escenario_2025.isna().sum().sum() > 0:
    print("\nADVERTENCIA: Hay nulos en el escenario 2025. Se imputarán en el pipeline.")

# Predicción 2025
pred_2025 = model.predict(escenario_2025)
escenario_2025['hc_estimado_2025'] = np.maximum(np.round(pred_2025), 0).astype(int)

print("\n=== PROYECCION 2025 POR AREA Y PUESTO ===")
print(
    escenario_2025[['area', 'puesto', 'headcount', 'hc_estimado_2025']]
    .sort_values(['area', 'puesto'])
    .to_string(index=False)
)

# Totales por área
resumen_area_2025 = (
    escenario_2025.groupby('area', as_index=False)
    .agg(
        hc_actual_2024=('headcount', 'sum'),
        hc_estimado_2025=('hc_estimado_2025', 'sum')
    )
)

resumen_area_2025['crecimiento_hc_abs'] = resumen_area_2025['hc_estimado_2025'] - resumen_area_2025['hc_actual_2024']
resumen_area_2025['crecimiento_hc_pct'] = np.where(
    resumen_area_2025['hc_actual_2024'] > 0,
    resumen_area_2025['crecimiento_hc_abs'] / resumen_area_2025['hc_actual_2024'],
    0
)

print("\n=== RESUMEN 2025 POR AREA ===")
print(resumen_area_2025.sort_values('area').to_string(index=False))


# ============================================================
# 10) SALIDA FINAL UTIL PARA NEGOCIO
# ============================================================
salida_final = escenario_2025[['anio', 'area', 'puesto', 'headcount', 'hc_estimado_2025']].copy()
salida_final = salida_final.rename(columns={
    'headcount': 'hc_actual_2024'
})

salida_final['delta_hc'] = salida_final['hc_estimado_2025'] - salida_final['hc_actual_2024']

print("\n=== SALIDA FINAL ===")
print(salida_final.sort_values(['area', 'puesto']).to_string(index=False))

Años disponibles en X: [np.int64(2022), np.int64(2023)]
Train años: [np.int64(2022)]
Test años: [np.int64(2023)]
Filas train: 24
Filas test: 24
=== METRICAS TEST ===
MAE : 0.88
RMSE: 1.95
R2  : 0.9590

=== RESULTADOS TEST ===
 anio      area         puesto  headcount  real_next_year  pred_next_year  error_abs
 2023 Comercial       Analista         22              25              25          0
 2023 Comercial    Coordinador          4               4               4          0
 2023 Comercial   Especialista         11              12              12          0
 2023 Comercial        Gerente          1               1               2          1
 2023  Finanzas       Analista         17              18              18          0
 2023  Finanzas    Coordinador          3               3               3          0
 2023  Finanzas   Especialista          8               9               9          0
 2023  Finanzas        Gerente          1               1               1          0
 2023 Log

In [4]:
salida_final.to_excel("salida_final.xlsx")